# Feature Reconstruction Engine

The trained LightGBM model expects 135 engineered features as input.

However, requiring end users to manually provide all model features would be impractical and significantly reduce usability.

To solve this problem, a Feature Reconstruction Engine is introduced.

The engine accepts a small set of business-friendly user inputs and automatically generates the full feature set required by the machine learning model.

### Objectives

- Reduce user input complexity
- Preserve model compatibility
- Support frontend deployment
- Enable prediction from real-world customer applications

### Workflow

User Inputs (~15 Features)
Feature Reconstruction Engine
135 Model Features
Prediction Pipeline
Risk Assessment
Recommendations and Explainability

In [ ]:
import joblib

feature_columns = joblib.load(
    "../models/feature_columns.pkl"
)

print(
    "Total Features:",
    len(feature_columns)
)

feature_columns[:30]

In [ ]:
for i, feature in enumerate(feature_columns):
    print(i, feature)

## User Input Schema Design

The Feature Reconstruction Engine is based on a reduced set of business-friendly customer inputs.

Rather than requesting all 135 model features, users provide a small number of financial and demographic attributes that are commonly available during a loan application.

These inputs are then transformed into the complete feature set required by the trained machine learning model.

### Planned User Inputs

- Age
- Gender
- Income
- Credit Score
- Loan Amount
- Annuity Amount
- Employment Years
- Number of Children
- Marital Status
- Education Level
- Housing Type
- Income Type
- Contract Type
- Own Car
- Own Property

The remaining model features are generated through feature engineering, business rules, credit-score mapping, one-hot encoding, and default-value assignment.

In [ ]:
USER_INPUT_SCHEMA = {

    "AGE_YEARS": 30,

    "GENDER": "M",

    "INCOME_TOTAL": 500000,

    "CREDIT_SCORE": 700,

    "AMT_CREDIT": 1000000,

    "AMT_ANNUITY": 50000,

    "EMPLOYMENT_YEARS": 5,

    "CNT_CHILDREN": 0,

    "MARITAL_STATUS": "Married",

    "EDUCATION": "Higher_education",

    "HOUSING_TYPE": "House_apartment",

    "INCOME_TYPE": "Working",

    "CONTRACT_TYPE": "Cash_loans",

    "OWN_CAR": True,

    "OWN_REALTY": True,

    "OWN_CAR_AGE": None,

    "HAS_PHONE": True,

    "HAS_EMAIL": True
}

USER_INPUT_SCHEMA

### Feature Vector Initialization

The first step of reconstruction is creating an empty feature vector containing all 135 model features.

Every feature is initialized before being populated through user inputs, derived values, one-hot encoding, credit score mapping, or default-value assignment.

This guarantees that the reconstructed feature set matches the exact structure expected by the trained LightGBM model.

In [ ]:
def initialize_feature_vector():

    feature_vector = {
        feature: 0
        for feature in feature_columns
    }

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

print(
    len(feature_vector)
)

list(feature_vector.items())[:10]

### Direct Feature Population

Certain model features correspond directly to information collected from the user application form.

These features are copied into the feature vector without additional transformation.

In [ ]:
def populate_direct_features(
    feature_vector,
    user_data
):

    feature_vector["CNT_CHILDREN"] = (
        user_data["CNT_CHILDREN"]
    )

    feature_vector["AMT_INCOME_TOTAL"] = (
        user_data["INCOME_TOTAL"]
    )

    feature_vector["AMT_CREDIT"] = (
        user_data["AMT_CREDIT"]
    )

    feature_vector["AMT_ANNUITY"] = (
        user_data["AMT_ANNUITY"]
    )

    feature_vector["FLAG_PHONE"] = int(
        user_data["HAS_PHONE"]
    )

    feature_vector["FLAG_EMAIL"] = int(
        user_data["HAS_EMAIL"]
    )

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_direct_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    k: feature_vector[k]
    for k in [
        "CNT_CHILDREN",
        "AMT_INCOME_TOTAL",
        "AMT_CREDIT",
        "AMT_ANNUITY",
        "FLAG_PHONE",
        "FLAG_EMAIL"
    ]
}

### Derived Feature Population

Several model features are not directly collected from users but can be calculated from existing inputs.

Examples include:

- DAYS_BIRTH
- DAYS_EMPLOYED
- AGE_YEARS
- EMPLOYMENT_YEARS
- CNT_FAM_MEMBERS
- FLAG_EMP_PHONE

These features preserve compatibility with the original Home Credit dataset while allowing users to provide intuitive business-friendly information.

In [ ]:
def populate_derived_features(
    feature_vector,
    user_data
):

    age_years = user_data["AGE_YEARS"]

    employment_years = user_data["EMPLOYMENT_YEARS"]

    feature_vector["AGE_YEARS"] = age_years

    feature_vector["EMPLOYMENT_YEARS"] = employment_years

    feature_vector["DAYS_BIRTH"] = -(
        age_years * 365
    )

    feature_vector["DAYS_EMPLOYED"] = -(
        employment_years * 365
    )

    feature_vector["CNT_FAM_MEMBERS"] = (
        user_data["CNT_CHILDREN"] + 2
    )

    feature_vector["FLAG_EMP_PHONE"] = (
        1 if employment_years > 0 else 0
    )

    feature_vector["FLAG_WORK_PHONE"] = (
        1 if employment_years > 0 else 0
    )

    feature_vector["FLAG_CONT_MOBILE"] = 1

    feature_vector["FLAG_MOBIL"] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_direct_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

feature_vector = populate_derived_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    k: feature_vector[k]
    for k in [
        "AGE_YEARS",
        "EMPLOYMENT_YEARS",
        "DAYS_BIRTH",
        "DAYS_EMPLOYED",
        "CNT_FAM_MEMBERS",
        "FLAG_EMP_PHONE"
    ]
}

### Credit Score Feature Generation

The Home Credit Default Risk dataset contains external risk assessment variables named:

- EXT_SOURCE_1
- EXT_SOURCE_2
- EXT_SOURCE_3

These variables are among the strongest predictors of default risk.

Since real users do not have access to these proprietary scores, the project uses a credit-score mapping strategy.

The user-provided credit score is transformed into synthetic EXT_SOURCE values using risk-aligned business rules.

This enables realistic risk estimation while maintaining a simple application experience.

In [ ]:
def credit_score_to_ext_sources(credit_score):

    credit_score = max(
        300,
        min(850, credit_score)
    )

    bands = [
    (300, 579, 0.05, 0.30, "Poor"),
    (580, 669, 0.30, 0.55, "Fair"),
    (670, 739, 0.55, 0.75, "Good"),
    (740, 799, 0.75, 0.90, "Very Good"),
    (800, 850, 0.90, 1.00, "Excellent")
]

    for score_min, score_max, ext_min, ext_max, category in bands:

        if score_min <= credit_score <= score_max:

            position = (
                credit_score - score_min
            ) / (
                score_max - score_min
            )

            base_score = (
                ext_min +
                position * (ext_max - ext_min)
            )

            ext_source_1 = max(
                0,
                min(1, base_score * 0.98)
            )

            ext_source_2 = max(
                0,
                min(1, base_score)
            )

            ext_source_3 = max(
                0,
                min(1, base_score * 1.02)
            )

            ext_source_mean = (
                ext_source_1 +
                ext_source_2 +
                ext_source_3
            ) / 3

            return {
                "EXT_SOURCE_1": round(ext_source_1, 4),
                "EXT_SOURCE_2": round(ext_source_2, 4),
                "EXT_SOURCE_3": round(ext_source_3, 4),
                "EXT_SOURCE_MEAN": round(ext_source_mean, 4),
                "CREDIT_PROFILE_CATEGORY": category
            }

In [ ]:
def populate_credit_score_features(
    feature_vector,
    user_data
):

    credit_features = credit_score_to_ext_sources(
        user_data["CREDIT_SCORE"]
    )

    feature_vector["EXT_SOURCE_1"] = (
        credit_features["EXT_SOURCE_1"]
    )

    feature_vector["EXT_SOURCE_2"] = (
        credit_features["EXT_SOURCE_2"]
    )

    feature_vector["EXT_SOURCE_3"] = (
        credit_features["EXT_SOURCE_3"]
    )

    feature_vector["EXT_SOURCE_MEAN"] = (
        credit_features["EXT_SOURCE_MEAN"]
    )

    feature_vector["EXT_SOURCE_1_MISSING"] = 0

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_direct_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

feature_vector = populate_derived_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

feature_vector = populate_credit_score_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    k: feature_vector[k]
    for k in [
        "EXT_SOURCE_1",
        "EXT_SOURCE_2",
        "EXT_SOURCE_3",
        "EXT_SOURCE_MEAN"
    ]
}

# One-Hot Feature Reconstruction

## Objective

The trained LightGBM model expects categorical variables in their one-hot encoded form rather than as raw business values.

For example, a user naturally enters values such as:

- Gender = Male
- Education = Higher Education
- Housing Type = House/Apartment
- Income Type = Working

However, during model training these categories were transformed into binary indicator features using one-hot encoding. Consequently, the model expects features such as:

- CODE_GENDER_M
- NAME_EDUCATION_TYPE_Higher_education
- NAME_HOUSING_TYPE_House___apartment
- NAME_INCOME_TYPE_Working

instead of the original categorical values.

## Reconstruction Strategy

The Feature Reconstruction Engine automatically converts business-friendly user inputs into the exact one-hot encoded representation used during model training.

Each categorical domain is reconstructed independently through dedicated encoder modules, including:

- Gender Encoder
- Property Ownership Encoder
- Contract Type Encoder
- Income Type Encoder
- Education Encoder
- Family Status Encoder
- Housing Type Encoder

This modular approach improves readability, maintainability, testing, and future extensibility while ensuring that the reconstructed feature vector exactly matches the schema expected by the trained model.

## Gender Feature Reconstruction

The trained model represents gender using one-hot encoded binary features instead of a raw categorical value.

The Feature Reconstruction Engine converts the business-friendly user input (`M` or `F`) into the corresponding encoded representation expected by the trained model.

During training, an additional category (`XNA`) existed for unknown or invalid gender values. Since the application only accepts valid user inputs, this feature is always reconstructed as `0`, ensuring consistency with the original training schema.

In [ ]:
def populate_gender_features(
    feature_vector,
    user_data
):

    gender = user_data["GENDER"]

    feature_vector["CODE_GENDER_M"] = (
        1 if gender == "M" else 0
    )

    feature_vector["CODE_GENDER_XNA"] = 0

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_gender_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    "CODE_GENDER_M":
        feature_vector["CODE_GENDER_M"],

    "CODE_GENDER_XNA":
        feature_vector["CODE_GENDER_XNA"]
}

In [ ]:
male_input = USER_INPUT_SCHEMA.copy()

male_input["GENDER"] = "M"

feature_vector = initialize_feature_vector()

feature_vector = populate_gender_features(
    feature_vector,
    male_input
)

print(feature_vector["CODE_GENDER_M"])
print(feature_vector["CODE_GENDER_XNA"])

In [ ]:
female_input = USER_INPUT_SCHEMA.copy()

female_input["GENDER"] = "F"

feature_vector = initialize_feature_vector()

feature_vector = populate_gender_features(
    feature_vector,
    female_input
)

print(feature_vector["CODE_GENDER_M"])
print(feature_vector["CODE_GENDER_XNA"])

## Property Ownership Feature Reconstruction

The trained model represents property ownership using one-hot encoded binary features rather than Boolean business inputs.

The Feature Reconstruction Engine converts the user's ownership information into the corresponding encoded features expected by the trained model.

Two independent ownership attributes are reconstructed:

- Vehicle Ownership
- Real Estate Ownership

Each feature is represented as a binary indicator where `1` denotes ownership and `0` denotes the absence of ownership. This transformation preserves the representation used during model training while allowing users to provide intuitive business-friendly inputs.

In [ ]:
def populate_property_features(
    feature_vector,
    user_data
):

    feature_vector["FLAG_OWN_CAR_Y"] = (
        1 if user_data["OWN_CAR"] else 0
    )

    feature_vector["FLAG_OWN_REALTY_Y"] = (
        1 if user_data["OWN_REALTY"] else 0
    )

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_property_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    "FLAG_OWN_CAR_Y":
        feature_vector["FLAG_OWN_CAR_Y"],

    "FLAG_OWN_REALTY_Y":
        feature_vector["FLAG_OWN_REALTY_Y"]
}

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["OWN_CAR"] = False
test_input["OWN_REALTY"] = False

feature_vector = initialize_feature_vector()

feature_vector = populate_property_features(
    feature_vector,
    test_input
)

{
    "FLAG_OWN_CAR_Y":
        feature_vector["FLAG_OWN_CAR_Y"],

    "FLAG_OWN_REALTY_Y":
        feature_vector["FLAG_OWN_REALTY_Y"]
}

## Education Feature Reconstruction

Education level was one-hot encoded during model training and contributes to the model's assessment of applicant risk.

Among all education categories, Higher Education was identified as one of the most influential categorical features through SHAP feature importance analysis.

The Feature Reconstruction Engine converts the user's selected education level into the corresponding one-hot encoded representation while ensuring that only one education category is activated at any time.

This guarantees consistency with the feature representation used during model training.

In [ ]:
def populate_education_features(
    feature_vector,
    user_data
):

    education_columns = [
        "NAME_EDUCATION_TYPE_Higher_education",
        "NAME_EDUCATION_TYPE_Incomplete_higher",
        "NAME_EDUCATION_TYPE_Lower_secondary",
        "NAME_EDUCATION_TYPE_Secondary___secondary_special"
    ]

    # Reset all education categories
    for column in education_columns:
        feature_vector[column] = 0

    education_mapping = {
        "Higher_education":
            "NAME_EDUCATION_TYPE_Higher_education",

        "Incomplete_higher":
            "NAME_EDUCATION_TYPE_Incomplete_higher",

        "Lower_secondary":
            "NAME_EDUCATION_TYPE_Lower_secondary",

        "Secondary_secondary_special":
            "NAME_EDUCATION_TYPE_Secondary___secondary_special"
    }

    selected_column = education_mapping.get(
        user_data["EDUCATION"]
    )

    if selected_column:
        feature_vector[selected_column] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_education_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    column: feature_vector[column]
    for column in [
        "NAME_EDUCATION_TYPE_Higher_education",
        "NAME_EDUCATION_TYPE_Incomplete_higher",
        "NAME_EDUCATION_TYPE_Lower_secondary",
        "NAME_EDUCATION_TYPE_Secondary___secondary_special"
    ]
}

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["EDUCATION"] = "Lower_secondary"

feature_vector = initialize_feature_vector()

feature_vector = populate_education_features(
    feature_vector,
    test_input
)

{
    column: feature_vector[column]
    for column in [
        "NAME_EDUCATION_TYPE_Higher_education",
        "NAME_EDUCATION_TYPE_Incomplete_higher",
        "NAME_EDUCATION_TYPE_Lower_secondary",
        "NAME_EDUCATION_TYPE_Secondary___secondary_special"
    ]
}

## Family Status Feature Reconstruction

Family status was one-hot encoded during model training and contributes to the model's assessment of applicant stability and repayment behaviour.

Among the available family status categories, "Married" was identified as one of the most influential categorical features through SHAP feature importance analysis.

The Feature Reconstruction Engine converts the applicant's selected family status into the corresponding one-hot encoded representation while ensuring that exactly one category is activated, maintaining consistency with the original training feature space.

In [ ]:
def populate_family_features(
    feature_vector,
    user_data
):

    family_columns = [
        "NAME_FAMILY_STATUS_Married",
        "NAME_FAMILY_STATUS_Separated",
        "NAME_FAMILY_STATUS_Single___not_married",
        "NAME_FAMILY_STATUS_Unknown",
        "NAME_FAMILY_STATUS_Widow"
    ]

    for column in family_columns:
        feature_vector[column] = 0

    family_mapping = {

        "Married":
            "NAME_FAMILY_STATUS_Married",

        "Separated":
            "NAME_FAMILY_STATUS_Separated",

        "Single_not_married":
            "NAME_FAMILY_STATUS_Single___not_married",

        "Unknown":
            "NAME_FAMILY_STATUS_Unknown",

        "Widow":
            "NAME_FAMILY_STATUS_Widow"

    }

    selected_column = family_mapping.get(
        user_data["MARITAL_STATUS"]
    )

    if selected_column:
        feature_vector[selected_column] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_family_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    column: feature_vector[column]
    for column in [
        "NAME_FAMILY_STATUS_Married",
        "NAME_FAMILY_STATUS_Separated",
        "NAME_FAMILY_STATUS_Single___not_married",
        "NAME_FAMILY_STATUS_Unknown",
        "NAME_FAMILY_STATUS_Widow"
    ]
}

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["MARITAL_STATUS"] = "Widow"

feature_vector = initialize_feature_vector()

feature_vector = populate_family_features(
    feature_vector,
    test_input
)

{
    column: feature_vector[column]
    for column in [
        "NAME_FAMILY_STATUS_Married",
        "NAME_FAMILY_STATUS_Separated",
        "NAME_FAMILY_STATUS_Single___not_married",
        "NAME_FAMILY_STATUS_Unknown",
        "NAME_FAMILY_STATUS_Widow"
    ]
}

## Contract Type Feature Reconstruction

Loan contract type was one-hot encoded during model training and represents the type of credit product requested by the applicant.

The Feature Reconstruction Engine converts the business-friendly contract type selected by the user into the corresponding encoded feature expected by the trained model.

The Home Credit dataset contains two primary contract types:

- Cash Loans
- Revolving Loans

Only one contract category is activated during reconstruction, ensuring consistency with the original training feature representation.

In [ ]:
def populate_contract_features(
    feature_vector,
    user_data
):

    feature_vector[
        "NAME_CONTRACT_TYPE_Revolving_loans"
    ] = (
        1
        if user_data["CONTRACT_TYPE"] == "Revolving_loans"
        else 0
    )

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_contract_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

feature_vector[
    "NAME_CONTRACT_TYPE_Revolving_loans"
]

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["CONTRACT_TYPE"] = "Revolving_loans"

feature_vector = initialize_feature_vector()

feature_vector = populate_contract_features(
    feature_vector,
    test_input
)

feature_vector[
    "NAME_CONTRACT_TYPE_Revolving_loans"
]

## Income Type Feature Reconstruction

Income source represents one of the applicant's primary financial characteristics and was one-hot encoded during model training.

The Feature Reconstruction Engine converts the user's selected income source into the corresponding encoded representation expected by the trained LightGBM model.

Each applicant belongs to exactly one income category; therefore, only one encoded feature is activated during reconstruction while all remaining categories remain inactive.

This reconstruction preserves the categorical representation used during model training and ensures consistency between user-friendly business inputs and the model's expected feature space.

In [ ]:
def populate_income_features(
    feature_vector,
    user_data
):

    income_columns = [

        "NAME_INCOME_TYPE_Commercial_associate",
        "NAME_INCOME_TYPE_Maternity_leave",
        "NAME_INCOME_TYPE_Pensioner",
        "NAME_INCOME_TYPE_State_servant",
        "NAME_INCOME_TYPE_Student",
        "NAME_INCOME_TYPE_Unemployed",
        "NAME_INCOME_TYPE_Working"

    ]

    for column in income_columns:
        feature_vector[column] = 0

    income_mapping = {

        "Commercial_associate":
            "NAME_INCOME_TYPE_Commercial_associate",

        "Maternity_leave":
            "NAME_INCOME_TYPE_Maternity_leave",

        "Pensioner":
            "NAME_INCOME_TYPE_Pensioner",

        "State_servant":
            "NAME_INCOME_TYPE_State_servant",

        "Student":
            "NAME_INCOME_TYPE_Student",

        "Unemployed":
            "NAME_INCOME_TYPE_Unemployed",

        "Working":
            "NAME_INCOME_TYPE_Working"

    }

    selected_column = income_mapping.get(
        user_data["INCOME_TYPE"]
    )

    if selected_column:
        feature_vector[selected_column] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_income_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    column: feature_vector[column]
    for column in [

        "NAME_INCOME_TYPE_Commercial_associate",
        "NAME_INCOME_TYPE_Maternity_leave",
        "NAME_INCOME_TYPE_Pensioner",
        "NAME_INCOME_TYPE_State_servant",
        "NAME_INCOME_TYPE_Student",
        "NAME_INCOME_TYPE_Unemployed",
        "NAME_INCOME_TYPE_Working"

    ]
}

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["INCOME_TYPE"] = "Student"

feature_vector = initialize_feature_vector()

feature_vector = populate_income_features(
    feature_vector,
    test_input
)

{
    column: feature_vector[column]
    for column in [

        "NAME_INCOME_TYPE_Commercial_associate",
        "NAME_INCOME_TYPE_Maternity_leave",
        "NAME_INCOME_TYPE_Pensioner",
        "NAME_INCOME_TYPE_State_servant",
        "NAME_INCOME_TYPE_Student",
        "NAME_INCOME_TYPE_Unemployed",
        "NAME_INCOME_TYPE_Working"

    ]
}

## Housing Type Feature Reconstruction

Housing type was one-hot encoded during model training and provides contextual information about the applicant's living arrangement.

The Feature Reconstruction Engine converts the user's selected housing type into the corresponding one-hot encoded representation expected by the trained model.

Since each applicant can belong to only one housing category, exactly one encoded feature is activated while all remaining housing categories remain inactive.

This reconstruction ensures complete compatibility with the feature space used during model training.

In [ ]:
def populate_housing_features(
    feature_vector,
    user_data
):

    housing_columns = [

        "NAME_HOUSING_TYPE_House___apartment",
        "NAME_HOUSING_TYPE_Municipal_apartment",
        "NAME_HOUSING_TYPE_Office_apartment",
        "NAME_HOUSING_TYPE_Rented_apartment",
        "NAME_HOUSING_TYPE_With_parents"

    ]

    for column in housing_columns:
        feature_vector[column] = 0

    housing_mapping = {

        "House_apartment":
            "NAME_HOUSING_TYPE_House___apartment",

        "Municipal_apartment":
            "NAME_HOUSING_TYPE_Municipal_apartment",

        "Office_apartment":
            "NAME_HOUSING_TYPE_Office_apartment",

        "Rented_apartment":
            "NAME_HOUSING_TYPE_Rented_apartment",

        "With_parents":
            "NAME_HOUSING_TYPE_With_parents"

    }

    selected_column = housing_mapping.get(
        user_data["HOUSING_TYPE"]
    )

    if selected_column:
        feature_vector[selected_column] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_housing_features(
    feature_vector,
    USER_INPUT_SCHEMA
)

{
    column: feature_vector[column]
    for column in [

        "NAME_HOUSING_TYPE_House___apartment",
        "NAME_HOUSING_TYPE_Municipal_apartment",
        "NAME_HOUSING_TYPE_Office_apartment",
        "NAME_HOUSING_TYPE_Rented_apartment",
        "NAME_HOUSING_TYPE_With_parents"

    ]
}

In [ ]:
test_input = USER_INPUT_SCHEMA.copy()

test_input["HOUSING_TYPE"] = "Rented_apartment"

feature_vector = initialize_feature_vector()

feature_vector = populate_housing_features(
    feature_vector,
    test_input
)

{
    column: feature_vector[column]
    for column in [

        "NAME_HOUSING_TYPE_House___apartment",
        "NAME_HOUSING_TYPE_Municipal_apartment",
        "NAME_HOUSING_TYPE_Office_apartment",
        "NAME_HOUSING_TYPE_Rented_apartment",
        "NAME_HOUSING_TYPE_With_parents"

    ]
}

## Intelligent Default Feature Reconstruction

Not every feature used during model training provides sufficient business value to justify asking the user for additional information.

To maintain a clean and user-friendly application while preserving compatibility with the trained model, the Feature Reconstruction Engine automatically reconstructs certain low-impact features using intelligent defaults derived from the training dataset.

This design follows three reconstruction strategies:

1. **User-Driven Reconstruction**
   - High-impact business features directly collected from the user.
   - Examples:
     - Age
     - Income
     - Credit Score
     - Employment
     - Education
     - Housing Type

2. **Derived Feature Reconstruction**
   - Features computed from user-provided information.
   - Examples:
     - EXT_SOURCE variables
     - EXT_SOURCE_MEAN
     - AGE_YEARS
     - EMPLOYMENT_YEARS
     - Credit Profile Category

3. **Intelligent Default Reconstruction**
   - Low-impact features that do not significantly improve user-facing decision making are automatically populated using statistically representative defaults from the training dataset.
   - This minimizes user effort while preserving compatibility with the original feature space expected by the trained LightGBM model.

This architecture balances prediction quality, usability, and production readiness by reducing unnecessary user inputs without sacrificing model compatibility.

## Applicant Accompaniment (Type Suite) Reconstruction

The Home Credit dataset records the applicant's accompaniment during the loan application process as a categorical feature.

Analysis of feature importance and business relevance indicated that this attribute has relatively low predictive value while offering limited practical value to end users.

Rather than requesting this information from every applicant, the Feature Reconstruction Engine automatically assigns the most representative category ("Unaccompanied"), which is the most common category observed during model training.

This approach reduces user friction while maintaining compatibility with the trained model's expected feature space.

In [ ]:
def populate_type_suite_features(
    feature_vector
):

    suite_columns = [

        "NAME_TYPE_SUITE_Family",
        "NAME_TYPE_SUITE_Group_of_people",
        "NAME_TYPE_SUITE_Other_A",
        "NAME_TYPE_SUITE_Other_B",
        "NAME_TYPE_SUITE_Spouse__partner",
        "NAME_TYPE_SUITE_Unaccompanied"

    ]

    for column in suite_columns:
        feature_vector[column] = 0

    # Default to the most common category
    feature_vector[
        "NAME_TYPE_SUITE_Unaccompanied"
    ] = 1

    return feature_vector

In [ ]:
feature_vector = initialize_feature_vector()

feature_vector = populate_type_suite_features(
    feature_vector,
)

{
    column: feature_vector[column]
    for column in [

        "NAME_TYPE_SUITE_Family",
        "NAME_TYPE_SUITE_Group_of_people",
        "NAME_TYPE_SUITE_Other_A",
        "NAME_TYPE_SUITE_Other_B",
        "NAME_TYPE_SUITE_Spouse__partner",
        "NAME_TYPE_SUITE_Unaccompanied"

    ]
}